In [123]:
import cx_Oracle
from tqdm import tqdm
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib

In [124]:
%matplotlib inline
sns.set_style('whitegrid')
from matplotlib import font_manager
# font_path='/usr/share/fonts/cjkuni-uming/uming.ttc'
font_path = '/usr/share/fonts/truetype/arphic/uming.ttc'
matplotlib.rcParams['font.family']=font_manager.FontProperties(fname=font_path).get_name()
matplotlib.rcParams['axes.unicode_minus']=False

In [125]:
import warnings
warnings.filterwarnings('ignore')

In [126]:
%matplotlib inline

### 批量处理所有股票

In [127]:
# 获取现金流量表中的年报数据
connection = cx_Oracle.connect('wind', 'wind', '10.6.60.114:1521/wind')
cursor = connection.cursor()
sql = "select * from ASHARECASHFLOW where REPORT_PERIOD like '%0331' and  STATEMENT_TYPE in (408001000,408005000,408027000,408028000,408036000,408045000)  order by REPORT_PERIOD,ACTUAL_ANN_DT"
cursor.execute(sql)
columns = [col[0] for col in cursor.description]
results = cursor.fetchall()
df1 = pd.DataFrame(results, columns=columns)
cursor.close()
connection.close()

# 获取现金流量表中的包含更正和调整数据
connection = cx_Oracle.connect('wind', 'wind', '10.6.60.114:1521/wind')
cursor = connection.cursor()
sql = "select * from ASHARECASHFLOW where REPORT_PERIOD like '%0331' and   STATEMENT_TYPE in (408001000,408004000,408050000,408029000,408031000,408037000,408046000)  order by REPORT_PERIOD,ACTUAL_ANN_DT"
cursor.execute(sql)
columns = [col[0] for col in cursor.description]
results = cursor.fetchall()
df3 = pd.DataFrame(results, columns=columns)
cursor.close()
connection.close()

In [128]:
start_date=df1['REPORT_PERIOD'].min()
end_date='20250626'
print(f"开始日期：{start_date},结束日期：{end_date}")

开始日期：20090331,结束日期：20250626


In [129]:
#生成日期序列
date_range=pd.date_range(start=start_date,end=end_date,freq='D')
df_dates=pd.DataFrame({'date':date_range})
#格式化日期
df_dates['Date']=df_dates['date'].dt.strftime("%Y%m%d")
df_dates['Date']=pd.to_datetime(df_dates['Date'])
df_dates=df_dates['Date']

In [130]:
df_dates

0      2009-03-31
1      2009-04-01
2      2009-04-02
3      2009-04-03
4      2009-04-04
          ...    
5927   2025-06-22
5928   2025-06-23
5929   2025-06-24
5930   2025-06-25
5931   2025-06-26
Name: Date, Length: 5932, dtype: datetime64[ns]

In [131]:
68001+73912+63921+97740

303574

In [132]:
print(f"df1:{df1.shape},df3:{df3.shape}")

df1:(68001, 126),df3:(126552, 126)


In [133]:
df3['STATEMENT_TYPE'].unique()

array(['408001000', '408004000', '408050000', '408029000'], dtype=object)

In [134]:
def prepare_data(df_merged,df4,period_offset):
    """
    准备原始季度数据：df_quarterly
    准备包含原始和调整数据：df_yoy
    period_offset:报告期偏移量
    """
    # 2. 计算实际报告间隔（动态替代固定3个月）
    df_merged['REPORT_PERIOD']=pd.to_datetime(df_merged['REPORT_PERIOD'])
    df_quarterly = df_merged.drop_duplicates(subset='REPORT_PERIOD').copy()
    df_quarterly['next_REPORT_PERIOD'] = df_quarterly['REPORT_PERIOD'].shift(period_offset)  # 获取下一次报告的实际日期
    df_quarterly['actual_interval'] = (df_quarterly['next_REPORT_PERIOD'] - df_quarterly['REPORT_PERIOD']).dt.days
    
    df_yoy=df4.copy()
    df_yoy['IS_ADJUSTED'] = df_yoy['STATEMENT_TYPE'].isin(['408004000', '408050000','408029000','408031000'])
    df_yoy['REPORT_PERIOD']=pd.to_datetime(df_yoy['REPORT_PERIOD'])
    df_yoy['ACTUAL_ANN_DT']=pd.to_datetime(df_yoy['ACTUAL_ANN_DT'])
    
    return df_quarterly,df_yoy

In [135]:
def create_next_period_dict(df_quarterly):
    """
    创建报告期与下一个报告期对应的公告日期的映射字典
    """
    # 创建一个字典，用于存储每个报告期对应的下一个报告期的ACTUAL_ANN_DT
    next_period_ann_dt = {}
    for idx, row in df_quarterly.iterrows():
        if pd.notna(row['next_REPORT_PERIOD']):
            # 查找下一个报告期对应的行
            next_period_rows = df_quarterly[df_quarterly['REPORT_PERIOD'] == row['next_REPORT_PERIOD']]
            if not next_period_rows.empty:
                next_period_ann_dt[row['REPORT_PERIOD']] = next_period_rows.iloc[0]['ACTUAL_ANN_DT']
    return next_period_ann_dt

In [136]:
def apply_adjusted_data(df_quarterly,df_yoy,next_period_ann_dt):
    """
    判断是否应用调整数据到季报数据中，并返回结果
    """
    result_df = df_quarterly.copy()
    
     # 遍历df_quarterly中的每一行
    for idx, orig_row in df_quarterly.iterrows():
        report_period = orig_row['REPORT_PERIOD']
        # 查找df_yoy中对应报告期的所有行（可能包含原始数据和多次调整数据）
        adj_rows = df_yoy[(df_yoy['REPORT_PERIOD'] == report_period) & (df_yoy['IS_ADJUSTED'] == True)]
        if not adj_rows.empty:
            # 获取下一个报告期的公告日期
            next_ann_dt = next_period_ann_dt.get(report_period)
            if next_ann_dt is not None:
                # 筛选出公告日期大于下一个报告期公告日期的调整数据
                valid_adj_rows = adj_rows[adj_rows['ACTUAL_ANN_DT'] <= next_ann_dt]
                if not valid_adj_rows.empty:
                    # 使用最新的有效调整数据
                    latest_adj_row = valid_adj_rows.iloc[-1]
                    # 更新结果DataFrame中的财务数据列
                    financial_columns = [col for col in result_df.columns]
                    
                    for col in financial_columns:
                        if col in latest_adj_row:
                            result_df.at[idx, col] = latest_adj_row[col]
    return result_df
    

In [137]:
def create_shifted_data(df_merged,result_df):
    """
    创建平移后的数据
    """
    # 3. 构建动态偏移映射表
    period_map = result_df.set_index('REPORT_PERIOD')['next_REPORT_PERIOD'].to_dict()
    
    # 4. 标记需要平移的列
    non_financial_columns = ['Date', 'REPORT_PERIOD']
    financial_columns = [col for col in df_merged.columns if col not in non_financial_columns]
    
    # 5. 创建平移后的DataFrame
    df_shifted = df_merged.copy()
    
    for col in financial_columns:
        # 为每个财务值找到下一次报告期的值
        df_quarterly_shifted = result_df.copy()
        df_quarterly_shifted['mapped_REPORT_PERIOD'] = df_quarterly_shifted['REPORT_PERIOD'].map(period_map)
        df_quarterly_shifted['mapped_REPORT_PERIOD'] = pd.to_datetime(df_quarterly_shifted['mapped_REPORT_PERIOD'])
        
        # 动态匹配
        temp_df = pd.merge_asof(
            df_merged[['REPORT_PERIOD']].sort_values('REPORT_PERIOD'),
            df_quarterly_shifted[['mapped_REPORT_PERIOD', col]]
                .dropna()
                .sort_values('mapped_REPORT_PERIOD'),
            left_on='REPORT_PERIOD',
            right_on='mapped_REPORT_PERIOD',
            direction='backward'
        )
        df_shifted[col] = temp_df[col].values
        
    df_shifted=df_shifted.dropna(subset=['S_INFO_WINDCODE'])

    #构建逆向字典
    reverse_period_map={v:k for k,v in period_map.items() if pd.notnull(v)}
    
    #映射回原始值
    df_shifted['REPORT_PERIOD']=df_shifted['REPORT_PERIOD'].map(reverse_period_map)
    
    return df_shifted

In [138]:
#获取所有股票代码
# stock_ids=df1['S_INFO_WINDCODE'].unique()
stock_ids=['600000.SH','600100.SH']
results=[]
results_shift=[]

In [139]:
for stock_id in tqdm(stock_ids,desc='Processing stocks'):
    #提取当前股票的年报数据并按实际公布日期排序
    stock_annual=df1[df1['S_INFO_WINDCODE']==stock_id].sort_values('ACTUAL_ANN_DT')
    
    stock_annual['ACTUAL_ANN_DT']=pd.to_datetime(stock_annual['ACTUAL_ANN_DT'])
    
    #合并两个数据框
    df_merged=pd.merge_asof(df_dates,stock_annual,left_on='Date',right_on='ACTUAL_ANN_DT',direction='backward')
    df_merged=df_merged.dropna(subset=['S_INFO_WINDCODE'])
    
    df4=df3[df3['S_INFO_WINDCODE']==stock_id]
    
    df_quarterly,df_yoy=prepare_data(df_merged, df4,period_offset=-2)
    
    # 创建一个字典，用于存储每个报告期对应的下一个报告期的ACTUAL_ANN_DT
    next_period_ann_dt = create_next_period_dict(df_quarterly)
                
    # 构造偏移时间轴数据时，调整数据应用
    result_df = apply_adjusted_data(df_quarterly,df_yoy,next_period_ann_dt)
    
    #创建平移后数据
    df_shifted=create_shifted_data(df_merged,result_df)

    results.append(df_merged)
    
    results_shift.append(df_shifted)
    
    # del stock_annual,df_quarterly,df_yoy,next_period_ann_dt,result_df
    
    

Processing stocks: 100%|██████████| 2/2 [00:01<00:00,  1.90it/s]


In [140]:
df4[['S_INFO_WINDCODE','ANN_DT','REPORT_PERIOD','STATEMENT_TYPE','CASH_RECP_SG_AND_RS','ACTUAL_ANN_DT']]

,S_INFO_WINDCODE,ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CASH_RECP_SG_AND_RS,ACTUAL_ANN_DT
1045,600100.SH,20090428,20090331,408001000,2.892389e+09,20090428
3232,600100.SH,20100428,20090331,408004000,2.892389e+09,20100428
5082,600100.SH,20100428,20100331,408001000,3.521403e+09,20100428
8099,600100.SH,20110430,20100331,408004000,3.521403e+09,20110430
10321,600100.SH,20110430,20110331,408001000,4.113076e+09,20110430
12909,600100.SH,20120428,20110331,408004000,4.113076e+09,20120428
15416,600100.SH,20120428,20120331,408001000,3.966100e+09,20120428
18035,600100.SH,20130427,20120331,408004000,3.966100e+09,20130427
20568,600100.SH,20130427,20130331,408001000,4.006719e+09,20130427
23129,600100.SH,20140430,20130331,408004000,4.068047e+09,20140430


In [141]:
df_merged

,Date,OBJECT_ID,S_INFO_WINDCODE,WIND_CODE,ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,...,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,OTHER_ACCOUNTS,OTHERS,MELT_MONEY_NET_INCREASE,SEC_FA_NET_FINA_INSTRUMENTS,OPDATE,OPMODE
28,2009-04-28,{BFE18D31-FDE9-4C29-8092-C7C70B810149},600100.SH,600100.SH,20090428,2009-03-31,408001000,CNY,2.892389e+09,29070289.75,...,NaN,None,NaN,NaN,NaN,None,NaN,NaN,2009-04-28 01:48:55,0
29,2009-04-29,{BFE18D31-FDE9-4C29-8092-C7C70B810149},600100.SH,600100.SH,20090428,2009-03-31,408001000,CNY,2.892389e+09,29070289.75,...,NaN,None,NaN,NaN,NaN,None,NaN,NaN,2009-04-28 01:48:55,0
30,2009-04-30,{BFE18D31-FDE9-4C29-8092-C7C70B810149},600100.SH,600100.SH,20090428,2009-03-31,408001000,CNY,2.892389e+09,29070289.75,...,NaN,None,NaN,NaN,NaN,None,NaN,NaN,2009-04-28 01:48:55,0
31,2009-05-01,{BFE18D31-FDE9-4C29-8092-C7C70B810149},600100.SH,600100.SH,20090428,2009-03-31,408001000,CNY,2.892389e+09,29070289.75,...,NaN,None,NaN,NaN,NaN,None,NaN,NaN,2009-04-28 01:48:55,0
32,2009-05-02,{BFE18D31-FDE9-4C29-8092-C7C70B810149},600100.SH,600100.SH,20090428,2009-03-31,408001000,CNY,2.892389e+09,29070289.75,...,NaN,None,NaN,NaN,NaN,None,NaN,NaN,2009-04-28 01:48:55,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5927,2025-06-22,{7F166CE2-2437-11F0-A12E-6CFE54208A61},600100.SH,600100.SH,20250429,2025-03-31,408001000,CNY,2.870946e+09,30821834.58,...,NaN,None,NaN,NaN,NaN,None,NaN,NaN,2025-05-13 17:38:03,1
5928,2025-06-23,{7F166CE2-2437-11F0-A12E-6CFE54208A61},600100.SH,600100.SH,20250429,2025-03-31,408001000,CNY,2.870946e+09,30821834.58,...,NaN,None,NaN,NaN,NaN,None,NaN,NaN,2025-05-13 17:38:03,1
5929,2025-06-24,{7F166CE2-2437-11F0-A12E-6CFE54208A61},600100.SH,600100.SH,20250429,2025-03-31,408001000,CNY,2.870946e+09,30821834.58,...,NaN,None,NaN,NaN,NaN,None,NaN,NaN,2025-05-13 17:38:03,1
5930,2025-06-25,{7F166CE2-2437-11F0-A12E-6CFE54208A61},600100.SH,600100.SH,20250429,2025-03-31,408001000,CNY,2.870946e+09,30821834.58,...,NaN,None,NaN,NaN,NaN,None,NaN,NaN,2025-05-13 17:38:03,1


In [142]:
results_shift[1]

,Date,OBJECT_ID,S_INFO_WINDCODE,WIND_CODE,ANN_DT,REPORT_PERIOD,STATEMENT_TYPE,CRNCY_CODE,CASH_RECP_SG_AND_RS,RECP_TAX_RENDS,...,SECURITIE_NETCASH_RECEIVED,OTHER_IMPAIR_LOSS_ASSETS,CREDIT_IMPAIRMENT_LOSS,RIGHT_USE_ASSETS_DEP,OTHER_ACCOUNTS,OTHERS,MELT_MONEY_NET_INCREASE,SEC_FA_NET_FINA_INSTRUMENTS,OPDATE,OPMODE
760,2011-04-30,{07490215-0061-4E13-A956-436453E836BC},600100.SH,600100.SH,20100428,2009-03-31,408004000,CNY,2.892389e+09,29070289.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010-04-27 23:51:26,0
761,2011-05-01,{07490215-0061-4E13-A956-436453E836BC},600100.SH,600100.SH,20100428,2009-03-31,408004000,CNY,2.892389e+09,29070289.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010-04-27 23:51:26,0
762,2011-05-02,{07490215-0061-4E13-A956-436453E836BC},600100.SH,600100.SH,20100428,2009-03-31,408004000,CNY,2.892389e+09,29070289.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010-04-27 23:51:26,0
763,2011-05-03,{07490215-0061-4E13-A956-436453E836BC},600100.SH,600100.SH,20100428,2009-03-31,408004000,CNY,2.892389e+09,29070289.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010-04-27 23:51:26,0
764,2011-05-04,{07490215-0061-4E13-A956-436453E836BC},600100.SH,600100.SH,20100428,2009-03-31,408004000,CNY,2.892389e+09,29070289.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010-04-27 23:51:26,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5927,2025-06-22,{631EF042-03CE-11EF-96FC-96A468826B84},600100.SH,600100.SH,20240427,2023-03-31,408004000,CNY,4.549129e+09,43143158.04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-26 21:17:53,0
5928,2025-06-23,{631EF042-03CE-11EF-96FC-96A468826B84},600100.SH,600100.SH,20240427,2023-03-31,408004000,CNY,4.549129e+09,43143158.04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-26 21:17:53,0
5929,2025-06-24,{631EF042-03CE-11EF-96FC-96A468826B84},600100.SH,600100.SH,20240427,2023-03-31,408004000,CNY,4.549129e+09,43143158.04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-26 21:17:53,0
5930,2025-06-25,{631EF042-03CE-11EF-96FC-96A468826B84},600100.SH,600100.SH,20240427,2023-03-31,408004000,CNY,4.549129e+09,43143158.04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-26 21:17:53,0
